In [1]:
import re
import time
import json
import math
import unicodedata
from dataclasses import dataclass
from typing import Any, Dict, List, Optional, Tuple
from bs4 import BeautifulSoup
from dataclasses import dataclass
from typing import Optional, Tuple

import pandas as pd
import numpy as np
import requests
from tqdm.auto import tqdm

import pyarrow
from pathlib import Path

try:
    from rapidfuzz import fuzz
    _HAS_RAPIDFUZZ = True
except Exception:
    _HAS_RAPIDFUZZ = False

In [2]:
#text normalization

def norm_text(s: Optional[str]) -> str:
    if s is None:
        return ""
    s = str(s)
    s = unicodedata.normalize("NFKD", s)
    s = s.replace("\u00ad", "")  # soft hyphen
    s = s.lower().strip()
    s = re.sub(r"\s+", " ", s)
    # remove surrounding quotes
    s = s.strip("\"'“”‘’")
    return s

def norm_title(s: Optional[str]) -> str:
    s = norm_text(s)
    # strip punctuation but keep spaces/alnum
    s = re.sub(r"[^a-z0-9\s]", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

def norm_doi(s: Optional[str]) -> str:
    if not s:
        return ""
    s = norm_text(s)
    s = s.replace("https://doi.org/", "").replace("http://doi.org/", "")
    s = s.replace("doi:", "").strip()
    return s


In [3]:
#safe requests helpers

SESSION = requests.Session()
SESSION.headers.update({
    "User-Agent": "Mozilla/5.0",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
})


def get_json(
    url: str,
    params: Optional[dict] = None,
    timeout=(5, 20),          # (connect_timeout, read_timeout)
    retries: int = 3,
    backoff: float = 1.5,
):
    last_err = None
    for i in range(retries):
        try:
            r = SESSION.get(url, params=params, timeout=timeout)

            if r.status_code == 429:
                time.sleep(backoff * (i + 1))
                continue

            r.raise_for_status()
            return r.json()    # <-- ALWAYS return dict here

        except Exception as e:
            last_err = e
            time.sleep(backoff * (i + 1))

    raise last_err



In [4]:
def row_title(row: dict) -> str:
    return (row.get("title_final") or row.get("title") or row.get("title_guess") or "").strip()

def row_year(row: dict) -> Optional[int]:
    y = row.get("year") or row.get("year_llm")
    try:
        return int(float(y)) if y is not None and str(y).strip() != "" else None
    except Exception:
        return None


In [5]:
def url_resolves(url: str, timeout=(5, 15), allow_redirects=True):
    try:
        r = SESSION.get(url, timeout=timeout, allow_redirects=allow_redirects)
        return (200 <= r.status_code < 400), r.url
    except Exception:
        return False, None


In [6]:
#for web references (not journal)

def url_exists(url: str, timeout=(5, 15)) -> bool:
    try:
        # HEAD is often blocked; use GET with streaming
        r = SESSION.get(url, timeout=timeout, allow_redirects=True, stream=True)
        ok = 200 <= r.status_code < 400
        r.close()
        return ok
    except Exception:
        return False

def fetch_page_title(url: str, timeout=(5, 20)) -> str:
    try:
        r = SESSION.get(url, timeout=timeout, allow_redirects=True)
        r.raise_for_status()
        soup = BeautifulSoup(r.text, "html.parser")
        t = soup.title.get_text(" ", strip=True) if soup.title else ""
        return t or ""
    except Exception:
        return ""

In [7]:
#DOI 

def doi_exists(doi: str):
    doi = norm_doi(doi)
    if not doi:
        return False, None

    url = f"https://doi.org/{doi}"
    try:
        # Don't follow redirects — just confirm DOI resolver returns a redirect target
        r = SESSION.get(url, timeout=(5, 15), allow_redirects=False)

        # Most valid DOIs return 301/302/303/307/308 with a Location header
        if r.status_code in (301, 302, 303, 307, 308) and r.headers.get("Location"):
            return True, r.headers["Location"]

        # Some DOIs may return 200 with content negotiation; treat 200 as exists too
        if 200 <= r.status_code < 300:
            return True, url

        return False, None

    except Exception:
        return False, None

def first_url(row: dict) -> str:
    # If it's a DOI reference, don't treat doi.org as a web URL here
    doi = norm_doi(row.get("doi","") or row.get("doi_llm","") or "")

    urls = row.get("urls", None)

    # Normalize urls into a plain Python list of strings
    if urls is None:
        url_list = []
    elif isinstance(urls, str):
        url_list = [urls]
    else:
        # handles list/tuple/set, numpy arrays, pandas Series, etc.
        try:
            url_list = list(urls)
        except Exception:
            url_list = []

    # Scan URLs first
    for u in url_list:
        if not isinstance(u, str):
            continue
        u = u.strip()
        if not u:
            continue
        if u.startswith(("http://", "https://")):
            # If DOI exists, skip doi.org URL here (DOI handled elsewhere)
            if doi and "doi.org/" in u:
                continue
            return u

    # Fallback to url_llm
    u = row.get("url_llm", None)
    if isinstance(u, str):
        u = u.strip()
        if u.startswith(("http://", "https://")):
            if doi and "doi.org/" in u:
                return ""
            return u

    return ""


@dataclass
class UrlCheck:
    ok: bool                 # reachable (2xx/3xx)
    status_code: Optional[int]
    final_url: Optional[str]
    content_type: Optional[str]
    note: str                # human-readable reason

def url_check(url: str, timeout=(5, 20), retries: int = 1, backoff: float = 1.5) -> UrlCheck:
    headers = {
        "User-Agent": "Mozilla/5.0",
        "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
    }

    last_exc = None
    for i in range(retries + 1):
        try:
            r = SESSION.get(url, headers=headers, timeout=timeout, allow_redirects=True, stream=True)
            status = r.status_code
            final_url = r.url
            ctype = r.headers.get("Content-Type")
            r.close()

            if 200 <= status < 400:
                return UrlCheck(True, status, final_url, ctype, "url reachable")

            if status == 403:
                # common WAF/bot block. Not evidence of non-existence.
                return UrlCheck(False, status, final_url, ctype, "url blocked (403)")

            if status in (429, 503, 502, 504):
                time.sleep(backoff * (i + 1))
                continue

            return UrlCheck(False, status, final_url, ctype, f"url returned HTTP {status}")

        except Exception as e:
            last_exc = e
            time.sleep(backoff * (i + 1))

    return UrlCheck(False, None, None, None, f"url error: {last_exc!r}")


# def fetch_page_title(url: str, timeout=(5, 20)) -> str:
#     try:
#         r = SESSION.get(url, timeout=timeout, allow_redirects=True)
#         r.raise_for_status()
#         soup = BeautifulSoup(r.text, "html.parser")
#         return soup.title.get_text(" ", strip=True) if soup.title else ""
#     except Exception:
#         return ""



In [8]:
#cross ref by doi

def crossref_by_doi(doi: str) -> Optional[Dict[str, Any]]:
    doi = norm_doi(doi)
    if not doi:
        return None
    data = get_json(f"https://api.crossref.org/works/{doi}", timeout=(5, 20), retries=2)
    it = data.get("message", {})
    return {
        "source": "crossref",
        "title": (it.get("title") or [""])[0],
        "doi": it.get("DOI", ""),
        "year": (it.get("issued", {}).get("date-parts", [[None]])[0][0]),
        "venue": (it.get("container-title") or [""])[0],
        "authors": [" ".join([a.get("given",""), a.get("family","")]).strip() for a in it.get("author", [])],
        "url": it.get("URL", "")
    }


In [9]:
#cross reference by url

def crossref_search(query: str, rows: int=5) -> List[Dict[str, Any]]:
    url = "https://api.crossref.org/works"
    data = get_json(url, params={"query.bibliographic": query, "rows": rows})
    items = data.get("message", {}).get("items", [])
    out = []
    for it in items:
        out.append({
            "source": "crossref",
            "title": (it.get("title") or [""])[0],
            "doi": it.get("DOI", ""),
            "year": (it.get("issued", {}).get("date-parts", [[None]])[0][0]),
            "venue": it.get("container-title", [""])[0] if it.get("container-title") else "",
            "authors": [" ".join([a.get("given",""), a.get("family","")]).strip() for a in it.get("author", [])],
            "url": it.get("URL", "")
        })
    return out


In [10]:
#pubmed search

def pubmed_esearch(term: str, retmax: int=5) -> List[str]:
    url = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi"
    params = {"db": "pubmed", "term": term, "retmode": "json", "retmax": retmax}
    data = get_json(url, params=params)
    return data.get("esearchresult", {}).get("idlist", [])

def pubmed_esummary(pmids: List[str]) -> List[Dict[str, Any]]:
    if not pmids:
        return []
    url = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esummary.fcgi"
    params = {"db": "pubmed", "id": ",".join(pmids), "retmode": "json"}
    data = get_json(url, params=params)
    result = data.get("result", {})
    out = []
    for pmid in pmids:
        it = result.get(pmid, {})
        out.append({
            "source": "pubmed",
            "title": it.get("title", ""),
            "doi": "",  # PubMed summary doesn't always expose DOI cleanly
            "year": int(it.get("pubdate","")[:4]) if str(it.get("pubdate","")[:4]).isdigit() else None,
            "venue": it.get("fulljournalname","") or it.get("source",""),
            "authors": [a.get("name","") for a in it.get("authors", [])],
            "url": f"https://pubmed.ncbi.nlm.nih.gov/{pmid}/",
            "pmid": pmid
        })
    return out


In [11]:
#openAlex search

def openalex_search(query: str, rows: int=5) -> List[Dict[str, Any]]:
    url = "https://api.openalex.org/works"
    params = {"search": query, "per-page": rows}
    data = get_json(url, params=params)
    items = data.get("results", [])
    out = []
    for it in items:
        out.append({
            "source": "openalex",
            "title": it.get("title",""),
            "doi": (it.get("doi","") or "").replace("https://doi.org/",""),
            "year": it.get("publication_year", None),
            "venue": (it.get("primary_location", {}) or {}).get("source", {}).get("display_name",""),
            "authors": [a.get("author",{}).get("display_name","") for a in it.get("authorships", [])],
            "url": it.get("id","")
        })
    return out


In [12]:


def openalex_by_doi(doi: str) -> Optional[Dict[str, Any]]:
    doi = norm_doi(doi)
    if not doi:
        return None
    url = "https://api.openalex.org/works/https://doi.org/" + doi
    it = get_json(url, timeout=(5, 20), retries=2)
    return {
        "source": "openalex",
        "title": it.get("title",""),
        "doi": (it.get("doi","") or "").replace("https://doi.org/",""),
        "year": it.get("publication_year", None),
        "venue": (it.get("primary_location", {}) or {}).get("source", {}).get("display_name",""),
        "authors": [a.get("author",{}).get("display_name","") for a in it.get("authorships", [])],
        "url": it.get("id","")
    }


In [13]:
#fuzzy matching

def title_similarity(a: str, b: str) -> float:
    a2, b2 = norm_title(a), norm_title(b)
    if not a2 or not b2:
        return 0.0
    if _HAS_RAPIDFUZZ:
        return float(fuzz.token_set_ratio(a2, b2)) / 100.0
    # fallback: simple token overlap
    ta, tb = set(a2.split()), set(b2.split())
    if not ta or not tb:
        return 0.0
    return len(ta & tb) / len(ta | tb)

def author_hit(row_authors: List[str], cand_authors: List[str]) -> float:
    ra = [norm_text(x) for x in (row_authors or []) if x]
    ca = [norm_text(x) for x in (cand_authors or []) if x]
    if not ra or not ca:
        return 0.0
    # first author last name match is strong
    def last(s): 
        toks = s.split()
        return toks[-1] if toks else ""
    r0 = last(ra[0])
    if r0 and any(last(x) == r0 for x in ca):
        return 1.0
    # any token overlap is weaker
    rlasts = set(last(x) for x in ra if last(x))
    clasts = set(last(x) for x in ca if last(x))
    return 0.6 if (rlasts & clasts) else 0.0

def year_score(row_year: Optional[int], cand_year: Optional[int]) -> float:
    if not row_year or not cand_year:
        return 0.0
    if row_year == cand_year:
        return 1.0
    if abs(row_year - cand_year) == 1:
        return 0.8
    if abs(row_year - cand_year) == 2:
        return 0.4
    return 0.0


In [14]:
#Evaluate Condidates

@dataclass
class VerifyResult:
    status: str
    score: int
    best_source: str
    best_match: Dict[str, Any]
    evidence_url: str
    notes: str

def _year_int(y):
    try:
        if y is None:
            return None
        s = str(y).strip()
        if not s:
            return None
        return int(float(s))   # handles "2018.0"
    except Exception:
        return None

def pick_best_match(row: Dict[str, Any], candidates: List[Dict[str, Any]]) -> VerifyResult:
    row_title = (row.get("title_final") or row.get("title") or row.get("title_guess") or "").strip()
    row_doi = norm_doi(row.get("doi","") or row.get("doi_llm","") or "")
    row_year = _year_int(row.get("year", None) or row.get("year_llm", None))
    row_authors = row.get("authors", []) or []
    if isinstance(row_authors, str):
        row_authors = [row_authors]

    best = None
    best_score = -1
    best_notes = ""

    for c in candidates:
        c_title = (c.get("title","") or "").strip()
        c_doi = norm_doi(c.get("doi","") or "")
        c_year = _year_int(c.get("year", None))

        s_title = title_similarity(row_title, c_title)
        s_year  = year_score(row_year, c_year)
        s_auth  = author_hit(row_authors, c.get("authors", []) or [])
        s_doi   = 1.0 if (row_doi and c_doi and row_doi == c_doi) else 0.0

        # guard against year-only false positives
        if s_title < 0.30 and s_auth == 0.0 and s_doi == 0.0:
            s_year = 0.0

        score = (0.55*s_title + 0.20*s_auth + 0.15*s_year + 0.60*s_doi)
        score_scaled = int(round(100 * min(1.0, score)))

        notes = f"title={s_title:.2f}, auth={s_auth:.2f}, year={s_year:.2f}, doi={s_doi:.2f}"

        if score_scaled > best_score:
            best_score = score_scaled
            best = c
            best_notes = notes

    if best is None:
        return VerifyResult("NO", 0, "", {}, "", "no candidates returned")

    # ---- HARD FAIL: DOI mismatch (prevents 'title match' verifying wrong work) ----
    best_doi = norm_doi(best.get("doi","") or "")
    if row_doi and best_doi and row_doi != best_doi:
        return VerifyResult(
            status="NO",
            score=0,
            best_source=best.get("source",""),
            best_match=best,
            evidence_url=best.get("url",""),
            notes=f"DOI mismatch: row={row_doi} vs candidate={best_doi} ({best_notes})"
        )

    # decide status thresholds
    if best_score >= 85 or (row_doi and best_doi == row_doi):
        status = "YES"
    elif best_score >= 65:
        status = "MAYBE"
    else:
        status = "NO"

    return VerifyResult(
        status=status,
        score=best_score,
        best_source=best.get("source",""),
        best_match=best,
        evidence_url=best.get("url",""),
        notes=best_notes
    )



In [15]:
#Build query variants

def _year_int(y):
    try:
        return int(float(y))
    except Exception:
        return None

def build_queries(row: Dict[str, Any]) -> List[str]:
    doi = norm_doi(row.get("doi",""))
    title = (row.get("title_final") or row.get("title") or row.get("title_guess") or "").strip()
    year = _year_int(row.get("year") or row.get("year_llm"))
    authors = row.get("authors", []) or []
    journal = (row.get("journal_or_source") or "").strip()

    queries: List[str] = []

    if doi:
        queries.append(doi)

    if title:
        queries.append(f"\"{title}\"")

        last = ""
        if authors:
            toks = norm_text(authors[0]).split()
            last = toks[-1] if toks else ""
            if len(last) < 2:
                last = ""

        if last and year:
            queries.append(f"{title} {last} {year}")
        elif last:
            queries.append(f"{title} {last}")

        if journal:
            queries.append(f"{title} {journal}")

    # de-dupe
    seen, out = set(), []
    for q in queries:
        qn = norm_text(q)
        if qn and qn not in seen:
            seen.add(qn)
            out.append(q)
    return out

def fetch_candidates_for_query(q: str) -> List[Dict[str, Any]]:
    cands = []
    # Crossref
    try:
        cands += crossref_search(q, rows=5)
    except Exception:
        pass
    # OpenAlex
    try:
        cands += openalex_search(q, rows=5)
    except Exception:
        pass
    # PubMed (only for some queries; quoted title works well)
    try:
        pmids = pubmed_esearch(q, retmax=5)
        cands += pubmed_esummary(pmids)
    except Exception:
        pass
    return cands

STRICT_DOI_IF_PRESENT = True

def verify_row(row: Dict[str, Any]) -> VerifyResult:
    """
    Verify a single reference row.

    Order of operations:
      1) DOI verification (Crossref/OpenAlex first, then doi.org resolver)
      2) URL verification (reachable/blocked/moved via URL_HANDLERS)
      3) Scholarly search fallback (queries -> candidates -> pick_best_match)
    """

    # -------------------------
    # 1) DOI path (ALWAYS first)
    # -------------------------
    doi = norm_doi(row.get("doi", "") or row.get("doi_llm", "") or "")

    if doi:
        # Try Crossref by DOI first (doesn't depend on doi.org resolving)
        try:
            cr = crossref_by_doi(doi)
            if cr and cr.get("title"):
                return VerifyResult(
                    status="YES",
                    score=100,
                    best_source="crossref",
                    best_match=cr,
                    evidence_url=cr.get("url", f"https://doi.org/{doi}"),
                    notes="doi verified (crossref)",
                )
        except Exception:
            pass

        # Try OpenAlex by DOI
        try:
            oa = openalex_by_doi(doi)
            if oa and oa.get("title"):
                return VerifyResult(
                    status="YES",
                    score=98,
                    best_source="openalex",
                    best_match=oa,
                    evidence_url=oa.get("url", f"https://doi.org/{doi}"),
                    notes="doi verified (openalex)",
                )
        except Exception:
            pass

        # Finally, try doi.org resolver (nice-to-have)
        try:
            ok, loc = doi_exists(doi)
            if ok:
                return VerifyResult(
                    status="YES",
                    score=95,
                    best_source="doi",
                    best_match={"doi": doi, "location": loc},
                    evidence_url=f"https://doi.org/{doi}",
                    notes="doi resolved (resolver)",
                )
        except Exception:
            pass

        # If the DOI is present but we cannot confirm it, DO NOT fall through
        # to title-search and accidentally confirm a different DOI.
        if STRICT_DOI_IF_PRESENT:
            return VerifyResult(
                status="NO",
                score=0,
                best_source="doi",
                best_match={"doi": doi},
                evidence_url=f"https://doi.org/{doi}",
                notes="DOI present but not confirmed; refusing title-only verification"
            )
        else:
            return VerifyResult(
                status="MAYBE",
                score=55,
                best_source="doi",
                best_match={"doi": doi},
                evidence_url=f"https://doi.org/{doi}",
                notes="doi present but not confirmed (apis/resolver failed)"
            )


    # -------------------------
    # 2) URL path (non-DOI refs)
    # -------------------------
    url = first_url(row)
    if url:
        chk = url_check(url)

        # If reachable, we're done
        if chk.ok:
            return VerifyResult(
                status="YES",
                score=90,
                best_source="url",
                best_match={"url": url, "final_url": chk.final_url, "status": chk.status_code},
                evidence_url=chk.final_url or url,
                notes=chk.note,
            )

        # Let handlers deal with special cases (403 blocked, CRAN 404 vignette, etc.)
        for h in URL_HANDLERS:
            vr = h(chk, url)
            if vr is not None:
                return vr

        # Otherwise it's a real failure
        return VerifyResult(
            status="NO",
            score=0,
            best_source="url",
            best_match={"url": url, "final_url": chk.final_url, "status": chk.status_code},
            evidence_url=chk.final_url or url,
            notes=chk.note,
        )

    # --------------------------------
    # 3) Scholarly search fallback path
    # --------------------------------
    queries = build_queries(row)
    if not queries:
        return VerifyResult(
            status="NO",
            score=0,
            best_source="",
            best_match={},
            evidence_url="",
            notes="no queries (missing title/doi/url)",
        )

    all_cands: List[Dict[str, Any]] = []
    for q in queries[:3]:
        try:
            all_cands.extend(fetch_candidates_for_query(q))
        except Exception:
            pass
        time.sleep(0.2)

    return pick_best_match(row, all_cands)



In [16]:
from urllib.parse import urlparse

def is_domain(url: str, domain: str) -> bool:
    try:
        host = urlparse(url).netloc.lower()
        return host == domain or host.endswith("." + domain)
    except Exception:
        return False

def cran_package_from_url(url: str) -> str:
    try:
        p = urlparse(url)
        parts = p.path.strip("/").split("/")
        # /web/packages/<pkg>/...
        if len(parts) >= 3 and parts[0] == "web" and parts[1] == "packages":
            return parts[2]
    except Exception:
        pass
    return ""

def handle_cran_vignette_404(chk, url):
    if chk.status_code != 404:
        return None

    host = (urlparse(url).netloc or "").lower()
    if "cran.r-project.org" not in host:
        return None

    pkg = cran_package_from_url(url)
    if not pkg:
        return None

    pkg_url = f"https://cran.r-project.org/package={pkg}"
    pkg_chk = url_check(pkg_url)
    if pkg_chk.ok:
        return VerifyResult(
            "YES", 85, "url",
            {"url": url, "package_url": pkg_url, "status": chk.status_code},
            pkg_url,
            "cran vignette moved; package page reachable"
        )
    return None

def handle_blocked_url(chk, url):
    if chk.status_code == 403:
        return VerifyResult(
            "MAYBE",
            60,
            "url",
            {"url": url, "final_url": chk.final_url, "status": chk.status_code},
            chk.final_url or url,
            "url blocked (403)"
        )
    return None

URL_HANDLERS = [
    handle_blocked_url, handle_cran_vignette_404,
]



In [17]:
#Running it over the dataframe

def run_verification(df: pd.DataFrame, limit: Optional[int]=None) -> pd.DataFrame:
    out = df.copy()
    # ensure authors is list-like
    if "authors" in out.columns:
        out["authors"] = out["authors"].apply(lambda x: x if isinstance(x, list) else ([] if pd.isna(x) else [str(x)]))

    idxs = out.index[:limit] if limit else out.index
    results = []

    for i in tqdm(idxs, total=len(idxs)):
        row = out.loc[i].to_dict()
        vr = verify_row(row)
        results.append((i, vr))

    out["verify_status"] = ""
    out["verify_score"] = np.nan
    out["verify_best_source"] = ""
    out["verify_best_match"] = ""
    out["verify_evidence_url"] = ""
    out["verify_notes"] = ""

    for i, vr in results:
        out.at[i, "verify_status"] = vr.status
        out.at[i, "verify_score"] = vr.score
        out.at[i, "verify_best_source"] = vr.best_source
        out.at[i, "verify_best_match"] = json.dumps(vr.best_match, ensure_ascii=False)[:2000]
        out.at[i, "verify_evidence_url"] = vr.evidence_url
        out.at[i, "verify_notes"] = vr.notes

    return out

# Example:
# verified_df = run_verification(df, limit=50)
# verified_df.head()


In [18]:
#import dataframe

file_path = Path("pHast_cam_DRAFT_revA.docx")
paper_id = file_path.stem

out_dir = Path("DataFrames")
out_dir.mkdir(parents=True, exist_ok=True)

out_path = out_dir / f"refs_extracted_{paper_id}.parquet"

df = pd.read_parquet(out_path)

In [19]:
i = df.index[1]
row = df.loc[i].to_dict()

print("DOI:", row.get("doi"))
print("TITLE_FINAL:", row.get("title_final"))
print("TITLE:", row.get("title"))

queries = build_queries(row)
print("QUERIES:", queries)

# fetch candidates for just the first query
q = queries[0] if queries else None
print("TEST QUERY:", q)

cands = fetch_candidates_for_query(q) if q else []
print("CANDIDATES:", len(cands))


DOI: 10.3390/s18124102
TITLE_FINAL: Over One-Year Long-Term Laboratory Tests of pH Electrodes in Terms of Industrial Applications Checking Stabilities of Their Parameters and Their Influence on Uncertainties of Measurements
TITLE: None
QUERIES: ['10.3390/s18124102', '"Over One-Year Long-Term Laboratory Tests of pH Electrodes in Terms of Industrial Applications Checking Stabilities of Their Parameters and Their Influence on Uncertainties of Measurements"', 'Over One-Year Long-Term Laboratory Tests of pH Electrodes in Terms of Industrial Applications Checking Stabilities of Their Parameters and Their Influence on Uncertainties of Measurements Sensors (Basel)']
TEST QUERY: 10.3390/s18124102
CANDIDATES: 7


In [20]:
%%time

verified_df = run_verification(df, limit=50)

  0%|          | 0/18 [00:00<?, ?it/s]

CPU times: total: 344 ms
Wall time: 10.9 s


In [21]:
verified_df

,ref_id,raw,year,doi,urls,title_guess,authors,title,journal_or_source,year_llm,...,url_llm,ref_id_is_missing,ref_id_is_duplicate,title_final,verify_status,verify_score,verify_best_source,verify_best_match,verify_evidence_url,verify_notes
0,1,"[1] Tiwari, R., & Mahalpure, G. S. (2025). A D...",2025.0,10.25082/JPBR.2024.02.001,[https://doi.org/10.25082/JPBR.2024.02.001],A Detailed Review of pH and its Applications,"[Tiwari, R., Mahalpure, G. S.]",A Detailed Review of pH and its Applications,Journal of Pharmaceutical and Biopharmaceutica...,2025.0,...,https://doi.org/10.25082/JPBR.2024.02.001,False,False,A Detailed Review of pH and its Applications,YES,100.0,crossref,"{""source"": ""crossref"", ""title"": ""A Detailed Re...",https://doi.org/10.25082/jpbr.2024.02.001,doi verified (crossref)
1,2,"[2] Wiora A, Wiora J. Over One-Year Long-Term ...",2018.0,10.3390/s18124102,[],Over One-Year Long-Term Laboratory Tests of pH...,"[Wiora A, Wiora J]",None,Sensors (Basel),2018.0,...,None,False,False,Over One-Year Long-Term Laboratory Tests of pH...,YES,100.0,crossref,"{""source"": ""crossref"", ""title"": ""Over One-Year...",https://doi.org/10.3390/s18124102,doi verified (crossref)
2,3,"[3] Monaghan MT, Brogan K, Lockington D, Rotch...",2020.0,10.1038/s41433-019-0737-z,[],Variability in measuring pH using litmus paper...,"[Monaghan MT, Brogan K, Lockington D, Rotchfor...",None,Eye (Lond),2020.0,...,None,False,False,Variability in measuring pH using litmus paper...,YES,100.0,crossref,"{""source"": ""crossref"", ""title"": ""Variability i...",https://doi.org/10.1038/s41433-019-0737-z,doi verified (crossref)
3,4,"[4] Yuto Nakamura, Masahiro Nagao, Machine lea...",2025.0,10.1093/chemle/upaf018,[https://doi.org/10.1093/chemle/upaf018],None,"[Yuto Nakamura, Masahiro Nagao]",None,Chemistry Letters,2025.0,...,https://doi.org/10.1093/chemle/upaf018,False,False,Machine learning-based pH quantification from ...,YES,100.0,crossref,"{""source"": ""crossref"", ""title"": ""Machine learn...",https://doi.org/10.1093/chemle/upaf018,doi verified (crossref)
4,5,"[5] Lawn JE, Cousens S, Zupan J. 4 million neo...",2005.0,10.1016/s0140-6736(05)71048-5,[https://doi.org/10.1016/s0140-6736(05],None,"[Lawn JE, Cousens S, Zupan J]",None,The Lancet,2005.0,...,None,False,False,4 million neonatal deaths: When? Where? Why? The,YES,100.0,crossref,"{""source"": ""crossref"", ""title"": ""4 million neo...",https://doi.org/10.1016/s0140-6736(05)71048-5,doi verified (crossref)
5,6,[6] Westgard QC. 2025 CLIA Acceptance Limits f...,2025.0,None,[https://westgard.com/clia-a-quality/quality-r...,"Westgard Quality Control, 2024,",[],"Westgard Quality Control, 2024",https://westgard.com/clia-a-quality/quality-re...,2025.0,...,https://westgard.com/clia-a-quality/quality-re...,False,False,"Westgard Quality Control, 2024",YES,90.0,url,"{""url"": ""https://westgard.com/clia-a-quality/q...",https://westgard.com/clia-a-quality/quality-re...,url reachable
6,7,"[7] Kirby, R., & Palamountain, K. (2020). pH m...",2020.0,None,[https://www.unicef.org/supply/media/2921/file...,"pH monitor: Target product profile (1st ed., v...","[Kirby, R., Palamountain, K.]","pH monitor: Target product profile (1st ed., v...",None,2020.0,...,https://www.unicef.org/supply/media/2921/file/...,False,False,"pH monitor: Target product profile (1st ed., v...",MAYBE,60.0,url,"{""url"": ""https://www.unicef.org/supply/media/2...",https://www.unicef.org/supply/media/2921/file/...,url blocked (403)
7,8,"[8] Gui, C., Wang, K., Li, C. et al. A CCD-bas...",2014.0,10.1186/1556-276X-9-57,[https://doi.org/10.1186/1556-276X-9-57],None,"[Gui, C., Wang, K.]",None,Nanoscale Research Letters,2014.0,...,https://doi.org/10.1186/1556-276X-9-57,False,False,"Li, C. et al. A CCD-based reader combined with...",YES,100.0,crossref,"{""source"": ""crossref"", ""title"": ""A CCD-based r...",https://doi.org/10.1186/1556-276x-9-57,doi verified (crossref)
8,9,"[9] Roda, A., Michelini, E., Zangheri, M., Di ...",2016.0,10.1016/j.trac.2015.10.019,

In [22]:
# save verified dataframe out to folder
out_dir = Path("DataFrames")
out_dir.mkdir(exist_ok=True)

verified_path = out_dir / "references_verified.parquet"

verified_df.to_parquet(verified_path, index=False)

In [23]:
i = 1
row = df.loc[i].to_dict()
print(row.keys())
print("doi field:", row.get("doi"))
print("DOI field:", row.get("DOI"))
print("title field:", row.get("title"))
print("Title field:", row.get("Title"))


dict_keys(['ref_id', 'raw', 'year', 'doi', 'urls', 'title_guess', 'authors', 'title', 'journal_or_source', 'year_llm', 'volume', 'issue', 'pages', 'doi_llm', 'url_llm', 'ref_id_is_missing', 'ref_id_is_duplicate', 'title_final'])
doi field: 10.3390/s18124102
DOI field: None
title field: None
Title field: None


In [24]:
x = get_json("https://api.crossref.org/works/10.3390/s18124102")
print(type(x), list(x.keys())[:4])

<class 'dict'> ['status', 'message-type', 'message-version', 'message']


In [25]:
print(doi_exists("10.3390/s18124102"))
print(crossref_by_doi("10.3390/s18124102"))
print(openalex_by_doi("10.3390/s18124102"))



(True, 'https://www.mdpi.com/1424-8220/18/12/4102')
{'source': 'crossref', 'title': 'Over One-Year Long-Term Laboratory Tests of pH Electrodes in Terms of Industrial Applications Checking Stabilities of Their Parameters and Their Influence on Uncertainties of Measurements', 'doi': '10.3390/s18124102', 'year': 2018, 'venue': 'Sensors', 'authors': ['Alicja Wiora', 'Józef Wiora'], 'url': 'https://doi.org/10.3390/s18124102'}
{'source': 'openalex', 'title': 'Over One-Year Long-Term Laboratory Tests of pH Electrodes in Terms of Industrial Applications Checking Stabilities of Their Parameters and Their Influence on Uncertainties of Measurements', 'doi': '10.3390/s18124102', 'year': 2018, 'venue': 'Sensors', 'authors': ['Alicja Wiora', 'Józef Wiora'], 'url': 'https://openalex.org/W2900730475'}


In [26]:
x = get_json("https://api.crossref.org/works/10.3390/s18124102", timeout=(5, 20), retries=1)
print(type(x), x if isinstance(x, tuple) else x.keys())


<class 'dict'> dict_keys(['status', 'message-type', 'message-version', 'message'])


In [27]:
i = df.index[1]
vr = verify_row(df.loc[i].to_dict())
print(vr)


VerifyResult(status='YES', score=100, best_source='crossref', best_match={'source': 'crossref', 'title': 'Over One-Year Long-Term Laboratory Tests of pH Electrodes in Terms of Industrial Applications Checking Stabilities of Their Parameters and Their Influence on Uncertainties of Measurements', 'doi': '10.3390/s18124102', 'year': 2018, 'venue': 'Sensors', 'authors': ['Alicja Wiora', 'Józef Wiora'], 'url': 'https://doi.org/10.3390/s18124102'}, evidence_url='https://doi.org/10.3390/s18124102', notes='doi verified (crossref)')


In [28]:
# If run_verification wrote columns back into df in-place, use df instead of dfv.
# df = df  # <-- change if needed

unconfirmed = verified_df[verified_df["verify_status"].isin(["NO", "MAYBE"])].copy()

cols = [
    "ref_id", "raw",
    "title_final", "authors", "year", "doi", "urls",
    "verify_status", "verify_score", "verify_best_source", "verify_notes", "verify_evidence_url"
]

unconfirmed[cols]


,ref_id,raw,title_final,authors,year,doi,urls,verify_status,verify_score,verify_best_source,verify_notes,verify_evidence_url
6,7,"[7] Kirby, R., & Palamountain, K. (2020). pH m...","pH monitor: Target product profile (1st ed., v...","[Kirby, R., Palamountain, K.]",2020.0,None,[https://www.unicef.org/supply/media/2921/file...,MAYBE,60.0,url,url blocked (403),https://www.unicef.org/supply/media/2921/file/...


In [29]:
def classify_row(r):
    has_doi = bool((r.get("doi") or "").strip())
    urls = r.get("urls") or []
    has_url = bool(urls) or (isinstance(r.get("url_llm"), str) and r["url_llm"].startswith("http"))
    title = (r.get("title_final") or r.get("title") or r.get("title_guess") or "").strip()
    has_title = bool(title)

    if has_doi:
        return "DOI present but not verified (network/API/format?)"
    if has_url:
        return "URL present but unreachable or blocked"
    if has_title:
        return "No DOI/URL; title-only search failed (needs query tuning/manual)"
    return "Missing title/doi/url (extraction issue)"

unconfirmed["failure_mode"] = unconfirmed.apply(classify_row, axis=1)
unconfirmed[["ref_id","verify_status","verify_score","failure_mode","verify_notes","verify_evidence_url","title_final","doi","urls"]]


,ref_id,verify_status,verify_score,failure_mode,verify_notes,verify_evidence_url,title_final,doi,urls
6,7,MAYBE,60.0,URL present but unreachable or blocked,url blocked (403),https://www.unicef.org/supply/media/2921/file/...,"pH monitor: Target product profile (1st ed., v...",None,[https://www.unicef.org/supply/media/2921/file...
